# PostgreSQL Data Loading

## Project
Amazon Germany Wireless Headphones Price and Competitor Intelligence

## Purpose
This notebook connects to the PostgreSQL database, loads the cleaned Amazon product dataset into the dimension and fact tables, records the ETL execution, and verifies the loaded data.

# Import the required libraries

In [1]:
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

print("Libraries imported successfully.")

Libraries imported successfully.


# Define project paths

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
ENV_FILE = PROJECT_ROOT / ".env"

CLEANED_CSV_FILE = (
    PROCESSED_DATA_DIR
    / "amazon_wireless_headphones_cleaned.csv"
)

print("Project root:", PROJECT_ROOT)
print("Environment file exists:", ENV_FILE.exists())
print("Cleaned CSV exists:", CLEANED_CSV_FILE.exists())

Project root: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project
Environment file exists: True
Cleaned CSV exists: True


# Load Database Credentials

## Read .env

In [3]:
load_dotenv(ENV_FILE)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

required_database_values = {
    "DB_HOST": DB_HOST,
    "DB_PORT": DB_PORT,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD
}

missing_database_values = [
    key
    for key, value in required_database_values.items()
    if not value
]

if missing_database_values:
    raise ValueError(
        f"Missing database settings in .env: "
        f"{missing_database_values}"
    )

print("Database credentials loaded successfully.")
print("Database host:", DB_HOST)
print("Database port:", DB_PORT)
print("Database name:", DB_NAME)
print("Database user:", DB_USER)
print("Database password is hidden.")

Database credentials loaded successfully.
Database host: localhost
Database port: 5432
Database name: amazon_competitor_bi
Database user: postgres
Database password is hidden.


## Create the PostgreSQL Connection
## Build the connection URL safely

In [5]:
database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

engine = create_engine(
    database_url,
    pool_pre_ping=True
)

print("SQLAlchemy engine created successfully.")

SQLAlchemy engine created successfully.


## Test the database connection

In [6]:
try:
    with engine.connect() as connection:
        connection_result = connection.execute(
            text("""
                SELECT
                    current_database() AS database_name,
                    current_user AS database_user,
                    version() AS postgresql_version;
            """)
        ).mappings().one()

    print("PostgreSQL connection successful.")
    print("Database:", connection_result["database_name"])
    print("User:", connection_result["database_user"])
    print(
        "PostgreSQL version:",
        connection_result["postgresql_version"].split(",")[0]
    )

except Exception as error:
    print("PostgreSQL connection failed.")
    raise error

PostgreSQL connection successful.
Database: amazon_competitor_bi
User: postgres
PostgreSQL version: PostgreSQL 18.4 on x86_64-windows


##  Load and Validate the Cleaned CSV
## Read the cleaned CSV

In [7]:
cleaned_df = pd.read_csv(
    CLEANED_CSV_FILE,
    encoding="utf-8-sig"
)

print("Cleaned CSV loaded successfully.")
print("Rows:", cleaned_df.shape[0])
print("Columns:", cleaned_df.shape[1])

display(cleaned_df.head())

Cleaned CSV loaded successfully.
Rows: 21
Columns: 20


,asin,product_title,brand,current_price,original_price,currency,discount_percentage,price_category,rating,rating_category,review_count,search_position,is_prime,is_sponsored,marketplace,search_query,product_url,image_url,extraction_date,extraction_timestamp
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,59.99,NaN,EUR,0.0,Mid-range,4.4,Good,463,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61WyJZWEhz...,2026-06-17,2026-06-17 15:54:20.345120
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,77.99,NaN,EUR,0.0,Premium,4.4,Good,128,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61wuPFetlI...,2026-06-17,2026-06-17 15:54:20.345120
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore,19.99,NaN,EUR,0.0,Budget,4.3,Good,108353,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/soundcore-Kabellose-Anpa...,https://m.media-amazon.com/images/I/5181ILcyQJ...,2026-06-17,2026-06-17 15:54:20.345120
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,49.00,NaN,EUR,0.0,Mid-range,4.2,Good,7427,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,https://m.media-amazon.com/images/I/61DFgTmj9x...,2026-06-17,2026-06-17 15:54:20.345120
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,29.00,NaN,EUR,0.0,Budget,4.5,Good,43479,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,https://m.media-amazon.com/images/I/61rFE093es...,2026-06-17,2026-06-17 15:54:20.345120


## Convert data types safely

In [8]:
cleaned_df["extraction_date"] = pd.to_datetime(
    cleaned_df["extraction_date"],
    errors="coerce"
).dt.date

cleaned_df["extraction_timestamp"] = pd.to_datetime(
    cleaned_df["extraction_timestamp"],
    errors="coerce"
)

cleaned_df["current_price"] = pd.to_numeric(
    cleaned_df["current_price"],
    errors="coerce"
)

cleaned_df["original_price"] = pd.to_numeric(
    cleaned_df["original_price"],
    errors="coerce"
)

cleaned_df["discount_percentage"] = pd.to_numeric(
    cleaned_df["discount_percentage"],
    errors="coerce"
)

cleaned_df["rating"] = pd.to_numeric(
    cleaned_df["rating"],
    errors="coerce"
)

cleaned_df["review_count"] = pd.to_numeric(
    cleaned_df["review_count"],
    errors="coerce"
).astype("Int64")

cleaned_df["search_position"] = pd.to_numeric(
    cleaned_df["search_position"],
    errors="coerce"
).astype("Int64")

cleaned_df["is_prime"] = (
    cleaned_df["is_prime"]
    .astype(str)
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
    .fillna(False)
)

cleaned_df["is_sponsored"] = (
    cleaned_df["is_sponsored"]
    .astype(str)
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
    .fillna(False)
)

print("Data types converted successfully.")

Data types converted successfully.


## Validate required values

In [9]:
required_columns = [
    "asin",
    "product_title",
    "brand",
    "current_price",
    "currency",
    "rating",
    "review_count",
    "marketplace",
    "search_query",
    "extraction_date",
    "extraction_timestamp"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in cleaned_df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required CSV columns: "
        f"{missing_required_columns}"
    )

required_null_counts = (
    cleaned_df[required_columns]
    .isna()
    .sum()
)

display(
    required_null_counts.to_frame(
        name="missing_values"
    )
)

if required_null_counts.sum() > 0:
    raise ValueError(
        "Required columns contain missing values."
    )

print("Required data validation passed.")

,missing_values
asin,0
product_title,0
brand,0
current_price,0
currency,0
rating,0
review_count,0
marketplace,0
search_query,0
extraction_date,0


Required data validation passed.


## Prepare the Dimension and Fact Data
## Create the product dimension DataFrame

In [10]:
dim_products_df = (
    cleaned_df[
        [
            "asin",
            "product_title",
            "brand",
            "product_url",
            "image_url",
            "marketplace",
            "search_query"
        ]
    ]
    .drop_duplicates(
        subset=["asin"]
    )
    .copy()
)

print("Dimension rows prepared:", len(dim_products_df))
display(dim_products_df.head())

Dimension rows prepared: 21


,asin,product_title,brand,product_url,image_url,marketplace,search_query
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61WyJZWEhz...,Amazon Germany,wireless headphones
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61wuPFetlI...,Amazon Germany,wireless headphones
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore,https://www.amazon.de/soundcore-Kabellose-Anpa...,https://m.media-amazon.com/images/I/5181ILcyQJ...,Amazon Germany,wireless headphones
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,https://m.media-amazon.com/images/I/61DFgTmj9x...,Amazon Germany,wireless headphones
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,https://m.media-amazon.com/images/I/61rFE093es...,Amazon Germany,wireless headphones


## Create the fact observation DataFrame

In [11]:
fact_observations_df = cleaned_df[
    [
        "asin",
        "current_price",
        "original_price",
        "currency",
        "discount_percentage",
        "price_category",
        "rating",
        "rating_category",
        "review_count",
        "search_position",
        "is_prime",
        "is_sponsored",
        "extraction_date",
        "extraction_timestamp"
    ]
].copy()

print(
    "Fact observation rows prepared:",
    len(fact_observations_df)
)

display(fact_observations_df.head())

Fact observation rows prepared: 21


,asin,current_price,original_price,currency,discount_percentage,price_category,rating,rating_category,review_count,search_position,is_prime,is_sponsored,extraction_date,extraction_timestamp
0,B0FSH42VZW,59.99,NaN,EUR,0.0,Mid-range,4.4,Good,463,<NA>,False,False,2026-06-17,2026-06-17 15:54:20.345120
1,B0FV3VX75R,77.99,NaN,EUR,0.0,Premium,4.4,Good,128,<NA>,False,False,2026-06-17,2026-06-17 15:54:20.345120
2,B0BTYCRJSS,19.99,NaN,EUR,0.0,Budget,4.3,Good,108353,<NA>,False,False,2026-06-17,2026-06-17 15:54:20.345120
3,B0DHL93XCN,49.00,NaN,EUR,0.0,Mid-range,4.2,Good,7427,<NA>,False,False,2026-06-17,2026-06-17 15:54:20.345120
4,B0BTJD6LCL,29.00,NaN,EUR,0.0,Budget,4.5,Good,43479,<NA>,False,False,2026-06-17,2026-06-17 15:54:20.345120


## Check Existing Database Rows
## Check current row counts

In [12]:
row_count_query = text("""
    SELECT
        'dim_products' AS table_name,
        COUNT(*) AS row_count
    FROM dim_products

    UNION ALL

    SELECT
        'fact_product_observations',
        COUNT(*)
    FROM fact_product_observations

    UNION ALL

    SELECT
        'etl_run_log',
        COUNT(*)
    FROM etl_run_log;
""")

with engine.connect() as connection:
    counts_before_df = pd.read_sql(
        row_count_query,
        connection
    )

print("Database row counts before loading:")
display(counts_before_df)

Database row counts before loading:


,table_name,row_count
0,dim_products,0
1,fact_product_observations,0
2,etl_run_log,0


## Load the Dimension Table

## Create the dimension upsert SQL 

In [14]:
dimension_upsert_sql = text("""
    INSERT INTO dim_products (
        asin,
        product_title,
        brand,
        product_url,
        image_url,
        marketplace,
        search_query
    )
    VALUES (
        :asin,
        :product_title,
        :brand,
        :product_url,
        :image_url,
        :marketplace,
        :search_query
    )
    ON CONFLICT (asin)
    DO UPDATE SET
        product_title = EXCLUDED.product_title,
        brand = EXCLUDED.brand,
        product_url = EXCLUDED.product_url,
        image_url = EXCLUDED.image_url,
        marketplace = EXCLUDED.marketplace,
        search_query = EXCLUDED.search_query,
        updated_at = CURRENT_TIMESTAMP;
""")

print("Dimension upsert SQL prepared.")

Dimension upsert SQL prepared.


## Convert dimension rows into records

In [15]:
dim_records = (
    dim_products_df
    .replace({np.nan: None})
    .to_dict(orient="records")
)

print("Dimension records ready:", len(dim_records))

Dimension records ready: 21


## Load the dimension records

In [17]:
with engine.begin() as connection:
    connection.execute(
        dimension_upsert_sql,
        dim_records
    )

print(
    f"{len(dim_records)} product dimension "
    f"records inserted or updated successfully."
)

21 product dimension records inserted or updated successfully.


## Load the Fact Table

## Create the fact insert SQL

In [18]:
fact_insert_sql = text("""
    INSERT INTO fact_product_observations (
        asin,
        current_price,
        original_price,
        currency,
        discount_percentage,
        price_category,
        rating,
        rating_category,
        review_count,
        search_position,
        is_prime,
        is_sponsored,
        extraction_date,
        extraction_timestamp
    )
    VALUES (
        :asin,
        :current_price,
        :original_price,
        :currency,
        :discount_percentage,
        :price_category,
        :rating,
        :rating_category,
        :review_count,
        :search_position,
        :is_prime,
        :is_sponsored,
        :extraction_date,
        :extraction_timestamp
    )
    ON CONFLICT (
        asin,
        extraction_timestamp
    )
    DO NOTHING;
""")

print("Fact insert SQL prepared.")

Fact insert SQL prepared.


## Convert fact rows safely into records

In [19]:
fact_records_df = fact_observations_df.copy()

fact_records_df = fact_records_df.astype(object).where(
    pd.notna(fact_records_df),
    None
)

fact_records = fact_records_df.to_dict(
    orient="records"
)

print("Fact records ready:", len(fact_records))

Fact records ready: 21


## Load the fact records

In [20]:
with engine.begin() as connection:
    fact_result = connection.execute(
        fact_insert_sql,
        fact_records
    )

print("Fact loading completed.")
print(
    "Rows inserted during this execution:",
    fact_result.rowcount
)

Fact loading completed.
Rows inserted during this execution: 21


## Insert an ETL Log Record
##  Prepare the ETL log values

In [22]:
etl_start_time = datetime.now()
etl_end_time = datetime.now()

records_extracted = 22
records_transformed = len(cleaned_df)
records_loaded = len(fact_records)

raw_files = list(
    (PROJECT_ROOT / "data" / "raw")
    .glob("*.json")
)

latest_raw_file = (
    max(
        raw_files,
        key=lambda file_path: file_path.stat().st_mtime
    )
    if raw_files
    else None
)

etl_log_record = {
    "pipeline_name": (
        "Amazon Germany Wireless Headphones ETL"
    ),
    "start_time": etl_start_time,
    "end_time": etl_end_time,
    "status": "SUCCESS",
    "records_extracted": records_extracted,
    "records_transformed": records_transformed,
    "records_loaded": records_loaded,
    "raw_file_path": (
        str(latest_raw_file)
        if latest_raw_file
        else None
    ),
    "processed_file_path": str(CLEANED_CSV_FILE),
    "error_message": None
}

print("ETL log record prepared.")

ETL log record prepared.


## Insert the ETL log

In [23]:
etl_log_insert_sql = text("""
    INSERT INTO etl_run_log (
        pipeline_name,
        start_time,
        end_time,
        status,
        records_extracted,
        records_transformed,
        records_loaded,
        raw_file_path,
        processed_file_path,
        error_message
    )
    VALUES (
        :pipeline_name,
        :start_time,
        :end_time,
        :status,
        :records_extracted,
        :records_transformed,
        :records_loaded,
        :raw_file_path,
        :processed_file_path,
        :error_message
    );
""")

with engine.begin() as connection:
    connection.execute(
        etl_log_insert_sql,
        etl_log_record
    )

print("ETL log inserted successfully.")

ETL log inserted successfully.


## Verify the Loaded Data
## Check row counts after loading

In [24]:
with engine.connect() as connection:
    counts_after_df = pd.read_sql(
        row_count_query,
        connection
    )

print("Database row counts after loading:")
display(counts_after_df)

Database row counts after loading:


,table_name,row_count
0,dim_products,21
1,fact_product_observations,21
2,etl_run_log,1


## Preview loaded products

In [25]:
product_preview_query = text("""
    SELECT
        asin,
        product_title,
        brand,
        marketplace
    FROM dim_products
    ORDER BY brand, product_title
    LIMIT 10;
""")

with engine.connect() as connection:
    product_preview_df = pd.read_sql(
        product_preview_query,
        connection
    )

display(product_preview_df)

,asin,product_title,brand,marketplace
0,B0DGHWD7CT,"Apple AirPods 4 Kabellose Kopfhörer, Bluetooth...",Apple,Amazon Germany
1,B0FDL13R3J,"Fachixy FC100 Headset mit Mikrofon, 2,4G Wirel...",Fachixy,Amazon Germany
2,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,Amazon Germany
3,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,Amazon Germany
4,B0DK3W2XK3,"JBL Tune Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,Amazon Germany
5,B0B5GP9FXN,"JBL Tune Flex TWS – Wasserdichte, True-Wireles...",JBL,Amazon Germany
6,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,Amazon Germany
7,B0FY2JJRB2,"Bluetooth Kopfhörer, In Ear Kopfhörer Kabellos...",Samsung,Amazon Germany
8,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,Amazon Germany
9,B0BTDX26B2,Sony WH-CH720N Kabelloser Bluetooth-Kopfhörer ...,Sony,Amazon Germany


## Preview loaded observations

In [26]:
observation_preview_query = text("""
    SELECT
        observation_id,
        asin,
        current_price,
        currency,
        rating,
        review_count,
        price_category,
        is_prime,
        is_sponsored,
        extraction_date
    FROM fact_product_observations
    ORDER BY observation_id
    LIMIT 10;
""")

with engine.connect() as connection:
    observation_preview_df = pd.read_sql(
        observation_preview_query,
        connection
    )

display(observation_preview_df)

,observation_id,asin,current_price,currency,rating,review_count,price_category,is_prime,is_sponsored,extraction_date
0,1,B0FSH42VZW,59.99,EUR,4.4,463,Mid-range,False,False,2026-06-17
1,2,B0FV3VX75R,77.99,EUR,4.4,128,Premium,False,False,2026-06-17
2,3,B0BTYCRJSS,19.99,EUR,4.3,108353,Budget,False,False,2026-06-17
3,4,B0DHL93XCN,49.00,EUR,4.2,7427,Mid-range,False,False,2026-06-17
4,5,B0BTJD6LCL,29.00,EUR,4.5,43479,Budget,False,False,2026-06-17
5,6,B0C3HCD34R,28.49,EUR,4.6,67076,Budget,False,False,2026-06-17
6,7,B0FY2JJRB2,10.99,EUR,4.4,1242,Budget,False,False,2026-06-17
7,8,B0FDL13R3J,35.66,EUR,4.3,374,Mid-range,False,False,2026-06-17
8,9,B0GLG2J9RM,25.64,EUR,4.3,22976,Budget,False,False,2026-06-17
9,10,B0GZGGDJ67,13.29,EUR,4.5,2209,Budget,False,False,2026-06-17


## Preview the ETL log

In [27]:
etl_log_preview_query = text("""
    SELECT
        run_id,
        pipeline_name,
        status,
        records_extracted,
        records_transformed,
        records_loaded,
        start_time,
        end_time
    FROM etl_run_log
    ORDER BY run_id DESC
    LIMIT 5;
""")

with engine.connect() as connection:
    etl_log_preview_df = pd.read_sql(
        etl_log_preview_query,
        connection
    )

display(etl_log_preview_df)

,run_id,pipeline_name,status,records_extracted,records_transformed,records_loaded,start_time,end_time
0,1,Amazon Germany Wireless Headphones ETL,SUCCESS,22,21,21,2026-06-17 18:42:56.492394,2026-06-17 18:42:56.492471


## Compare CSV and Database Counts
## Validate the loaded row counts

In [28]:
with engine.connect() as connection:
    database_product_count = connection.execute(
        text(
            "SELECT COUNT(*) FROM dim_products;"
        )
    ).scalar_one()

    database_observation_count = connection.execute(
        text(
            """
            SELECT COUNT(*)
            FROM fact_product_observations;
            """
        )
    ).scalar_one()

validation_summary = pd.DataFrame({
    "validation": [
        "Cleaned CSV rows",
        "Unique CSV products",
        "Database product rows",
        "Database observation rows"
    ],
    "result": [
        len(cleaned_df),
        cleaned_df["asin"].nunique(),
        database_product_count,
        database_observation_count
    ]
})

display(validation_summary)

,validation,result
0,Cleaned CSV rows,21
1,Unique CSV products,21
2,Database product rows,21
3,Database observation rows,21


## Confirm all ASINs loaded

In [29]:
with engine.connect() as connection:
    database_asins_df = pd.read_sql(
        text("SELECT asin FROM dim_products;"),
        connection
    )

csv_asins = set(
    cleaned_df["asin"]
    .dropna()
    .astype(str)
)

database_asins = set(
    database_asins_df["asin"]
    .dropna()
    .astype(str)
)

missing_in_database = sorted(
    csv_asins - database_asins
)

extra_in_database = sorted(
    database_asins - csv_asins
)

print("Missing ASINs in database:", missing_in_database)
print("Extra ASINs in database:", extra_in_database)

if not missing_in_database:
    print("All cleaned products were loaded successfully.")

Missing ASINs in database: []
Extra ASINs in database: []
All cleaned products were loaded successfully.
